## **make_pipeline:**
`sklearn.pipeline.make_pipeline` is a utility function in scikit-learn used to create a `Pipeline` object without explicitly naming the steps. It automatically generates step names based on the lowercase version of the estimator's name.

### Syntax
```python
from sklearn.pipeline import make_pipeline
pipeline = make_pipeline(transformer_1, transformer_2, estimator)
```

### Key Differences: `Pipeline` vs. `make_pipeline`

| Feature | `Pipeline` | `make_pipeline` |
| :--- | :--- | :--- |
| **Step Naming** | Manual (e.g., `('scaler', StandardScaler())`) | Automatic (e.g., `StandardScaler()`) |
| **Verbosity** | More explicit/verbose | More concise/shorthand |
| **Use Case** | When specific step names are required for grid searches | Rapid prototyping and quick workflows |

### Example Usage
```python
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# Create a pipeline: Scale data -> Reduce dimensions -> Classify
pipe = make_pipeline(StandardScaler(), PCA(n_components=2), LogisticRegression())

# The pipeline can be treated as a single estimator
pipe.fit(X_train, y_train)
predictions = pipe.predict(X_test)

# Accessing steps (names are derived from class names)
# pipe.steps -> [('standardscaler', StandardScaler()), ('pca', PCA()), ('logisticregression', LogisticRegression())]
```

### Benefits
1.  **Prevents Data Leakage:** By wrapping transformations and the estimator, `fit` and `transform` are applied correctly during cross-validation (the scaler is fit only on the training folds).
2.  **Code Cleanliness:** Reduces boilerplate code when building complex workflows.
3.  **Hyperparameter Tuning:** Allows you to perform a `GridSearchCV` over the entire workflow (e.g., tuning both the PCA components and the Logistic Regression penalty simultaneously).

## Example:

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold

from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA

In [11]:
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

In [ ]:
# Create make_pipeline
pipe_lr = make_pipeline(StandardScaler(),
                        PCA(n_components=2),
                        LogisticRegression(solver='lbfgs', random_state=42))

In [ ]:
# fit the training data
pipe_lr.fit(X_train, y_train)

,steps,"[('standardscaler', ...), ('pca', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,n_components,2
,copy,True
,whiten,False
,svd_solver,'auto'


In [ ]:
# make prediction base on the X_test
# don't need to use the '.transform()' for both StandardScaler() and PCA()
y_pred = pipe_lr.predict(X_test)

In [16]:
# Accuracy
pipe_lr.score(X_test, y_test)

0.9

## **PCA - Principle Component Analysis:**

### 1. Basic Implementation
To use Principal Component Analysis (PCA), you typically follow these steps: instantiate the model, `fit` it to your data, and `transform` the data into the new lower-dimensional space.

```python
from sklearn.decomposition import PCA
from sklearn.datasets import load_iris

# Load sample data
data = load_iris()
X = data.data

# 1. Initialize PCA (specify number of components)
pca = PCA(n_components=2)

# 2. Fit and Transform the data
X_reduced = pca.fit_transform(X)

print(f"Original shape: {X.shape}")
print(f"Reduced shape: {X_reduced.shape}")
```

### 2. Determining the Number of Components
Instead of guessing the number of components, you can pass a float between `0.0` and `1.0` to `n_components`. This tells PCA to select the number of components required to retain that percentage of **explained variance**.

```python
# Retain 95% of the variance
pca = PCA(n_components=0.95)
X_reduced = pca.fit_transform(X)
```

### 3. Essential Pre-processing: Scaling
**PCA is extremely sensitive to the scale of features.** Because PCA seeks to maximize variance, features with larger numerical ranges will dominate the components. You should **always** scale your data (usually with `StandardScaler`) before applying PCA.

```python
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# The correct workflow: Scale -> PCA
pipeline = make_pipeline(StandardScaler(), PCA(n_components=2))
X_reduced = pipeline.fit_transform(X)
```

### 4. Analyzing Results
After fitting, you can inspect how much information was retained using `explained_variance_ratio_`.

```python
pca = PCA().fit(X) # Fit without reducing to see all components
print(pca.explained_variance_ratio_)

# Example output: [0.72, 0.22, 0.05, 0.01]
# This means the 1st component explains 72% of the variance.
```

### Summary Checklist
1.  **Standardize:** Use `StandardScaler` first.
2.  **Choose $k$:** Decide on `n_components` (either a fixed integer or a variance percentage).
3.  **Fit/Transform:** Use `fit_transform` on training data; use `transform` on test data to avoid data leakage.
4.  **Evaluate:** Check `explained_variance_ratio_` to ensure you haven't lost too much information.

In [21]:
from sklearn.decomposition import PCA
from sklearn.datasets import load_iris

from sklearn.model_selection import train_test_split

In [32]:
# load data and split to train and test
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [34]:
# initialize PCA with 2 dimentions
pca = PCA(n_components=2)

# fit the pca to X_train
pca_fit = pca.fit(X_train)

# apply PCA to both X_train and X_test separately
X_train_reduced = pca_fit.transform(X_train)
X_test_reduced = pca_fit.transform(X_test)

In [36]:
# provide the percentage of explained by each feature
pca.explained_variance_ratio_

array([0.91959926, 0.05714377])